In [1]:
import pandas as pd
import sodapy
print("Environment is ready")

Environment is ready


In [2]:
from sodapy import Socrata

client = Socrata("data.cityofchicago.org", None, timeout=60)

# tiny test pull — just 5 rows
test = client.get("ijzp-q8t2", limit=5)
test_df = pd.DataFrame.from_records(test)
print(test_df.columns.tolist())
test_df.head()

['id', 'case_number', 'date', 'block', 'iucr', 'primary_type', 'description', 'location_description', 'arrest', 'domestic', 'beat', 'district', 'ward', 'community_area', 'fbi_code', 'x_coordinate', 'y_coordinate', 'year', 'updated_on', 'latitude', 'longitude', 'location', ':@computed_region_awaf_s7ux', ':@computed_region_6mkv_f3dw', ':@computed_region_vrxf_vc4k', ':@computed_region_bdys_3d7i', ':@computed_region_43wa_7qmu', ':@computed_region_rpca_8um6', ':@computed_region_d9mm_jgwp', ':@computed_region_d3ds_rm58', ':@computed_region_8hcu_yrd4']


,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,...,location,:@computed_region_awaf_s7ux,:@computed_region_6mkv_f3dw,:@computed_region_vrxf_vc4k,:@computed_region_bdys_3d7i,:@computed_region_43wa_7qmu,:@computed_region_rpca_8um6,:@computed_region_d9mm_jgwp,:@computed_region_d3ds_rm58,:@computed_region_8hcu_yrd4
0,14200649,JK258092,2026-05-14T00:00:00.000,013XX W FARGO AVE,4387,OTHER OFFENSE,VIOLATE ORDER OF PROTECTION,APARTMENT,False,True,...,"{'latitude': '42.017182802', 'longitude': '-87...",3,21853,10,357,5,9,11,32,49
1,14197966,JK255875,2026-05-14T00:00:00.000,049XX N LINCOLN AVE,0560,ASSAULT,SIMPLE,RESIDENCE,False,False,...,"{'latitude': '41.971187305', 'longitude': '-87...",13,21849,6,28,24,20,2,61,40
2,14200951,JK259356,2026-05-14T00:00:00.000,037XX W 76TH PL,1153,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,RESIDENCE,False,False,...,"{'latitude': '41.754325337', 'longitude': '-87...",6,4300,69,571,30,8,13,209,18
3,14202680,JK261478,2026-05-14T00:00:00.000,004XX E 44TH ST,1153,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,RESIDENCE,False,False,...,"{'latitude': '41.814843999', 'longitude': '-87...",12,4301,4,161,9,36,24,114,8
4,14200184,JK258546,2026-05-14T00:00:00.000,024XX W FOSTER AVE,0810,THEFT,OVER $500,STREET,False,False,...,"{'latitude': '41.975884204', 'longitude': '-87...",46,21849,6,28,24,20,2,57,40


In [4]:
import pandas as pd
import time
from sodapy import Socrata

client = Socrata("data.cityofchicago.org", None, timeout=120)

all_rows = []
offset = 0
page_size = 50_000
max_retries = 5

while True:
    # try the same page up to max_retries times before giving up
    for attempt in range(max_retries):
        try:
            page = client.get(
                "ijzp-q8t2",
                where="date >= '2019-01-01T00:00:00'",
                limit=page_size,
                offset=offset,
                order="date"
            )
            break  # success — exit the retry loop
        except Exception as e:
            wait = 10 * (attempt + 1)   # back off: 10s, 20s, 30s...
            print(f"  page at offset {offset:,} failed ({type(e).__name__}), "
                  f"retry {attempt + 1}/{max_retries} in {wait}s")
            time.sleep(wait)
    else:
        # all retries exhausted
        raise RuntimeError(f"Failed at offset {offset:,} after {max_retries} retries")

    if not page:
        break
    all_rows.extend(page)
    offset += page_size
    print(f"Pulled {len(all_rows):,} rows so far...")

df = pd.DataFrame.from_records(all_rows)
print(f"\nDone. Total: {len(df):,} rows")

Pulled 50,000 rows so far...
Pulled 100,000 rows so far...
Pulled 150,000 rows so far...
Pulled 200,000 rows so far...
Pulled 250,000 rows so far...
Pulled 300,000 rows so far...
Pulled 350,000 rows so far...
Pulled 400,000 rows so far...
Pulled 450,000 rows so far...
Pulled 500,000 rows so far...
Pulled 550,000 rows so far...
Pulled 600,000 rows so far...
Pulled 650,000 rows so far...
Pulled 700,000 rows so far...
Pulled 750,000 rows so far...
Pulled 800,000 rows so far...
Pulled 850,000 rows so far...
Pulled 900,000 rows so far...
Pulled 950,000 rows so far...
Pulled 1,000,000 rows so far...
Pulled 1,050,000 rows so far...
Pulled 1,100,000 rows so far...
Pulled 1,150,000 rows so far...
Pulled 1,200,000 rows so far...
Pulled 1,250,000 rows so far...
Pulled 1,300,000 rows so far...
Pulled 1,350,000 rows so far...
Pulled 1,400,000 rows so far...
Pulled 1,450,000 rows so far...
Pulled 1,500,000 rows so far...


KeyboardInterrupt: 

In [5]:
df = pd.DataFrame.from_records(all_rows)
print(f"Recovered {len(df):,} rows")
print(df["date"].min(), "→", df["date"].max())

Recovered 1,500,000 rows
2019-01-01T00:00:00.000 → 2025-03-29T13:00:00.000


In [6]:
print(df["primary_type"].nunique(), "crime types")
print(df[["latitude", "longitude"]].isna().sum())

33 crime types
latitude     23819
longitude    23819
dtype: int64


In [7]:
df.to_parquet("../data/crimes_raw.parquet", index=False)
print("Saved to ../data/crimes_raw.parquet")

OSError: Cannot save file into a non-existent directory: '../data'

In [8]:
import os
print("Notebook is running from:", os.getcwd())
print("Contents here:", os.listdir("."))

Notebook is running from: /Users/mallikachourasia/Documents/chicago-crime-analysis
Contents here: ['01_ingest.ipynb', 'venv', 'data', 'notebooks']


In [9]:
df.to_parquet("data/crimes_raw.parquet", index=False)
print("Saved to data/crimes_raw.parquet")

Saved to data/crimes_raw.parquet


In [10]:
import os
print(os.listdir("data"))
size_mb = os.path.getsize("data/crimes_raw.parquet") / 1e6
print(f"File size: {size_mb:.1f} MB")

['crimes_raw.parquet']
File size: 93.9 MB
